# Módulo 00 — internals: a série inteira, 2019–2024

O [walkthrough](srag_silver_walkthrough.ipynb) narra o tratamento em um ano. Este
caderno faz a pergunta que só a série responde: **o que muda entre os anos?**

Ele existe porque medir num ano e afirmar sobre os seis foi o erro que mais se
repetiu na construção deste módulo — checkboxes, ausência de comorbidade, idade,
`CO_DETEC`. A correção não foi escrever com mais cuidado; foi separar em dois
cadernos, e este varre sempre os seis.

São ~2,2 GB de microdados. O caderno faz **uma passagem só** sobre eles,
coletando tudo de que precisa, e apaga cada arquivo que ele mesmo baixou.

In [1]:
%pip install -q pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import pathlib
import subprocess
import sys
import tempfile
import urllib.request

import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.width", 120)

S3 = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG"
DATA = pathlib.Path.home() / "Documents/srag-data"
LOCAL = DATA.exists()
if not LOCAL:
    DATA = pathlib.Path(tempfile.gettempdir()) / "srag-data"
    DATA.mkdir(parents=True, exist_ok=True)

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "srag_silver.py").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "srag_silver.py").exists():
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))
import srag_silver as S

ANOS = list(range(2019, 2025))
print("microdados locais" if LOCAL else "serão baixados e apagados, um ano por vez")

microdados locais


## A passagem única

Uma leitura por ano, com projeção de colunas — nunca o arquivo inteiro em
memória, e nunca o mesmo ano baixado duas vezes. Tudo o que as seções seguintes
mostram sai deste laço.

In [3]:
# os conjuntos vêm do contrato — o notebook não redigita domínio nenhum
DOMINIOS = {c: list(S.DOMAINS[c]) for c in
            ["EVOLUCAO", "CS_SEXO", "UTI", "SUPORT_VEN", "CLASSI_FIN", "CRITERIO",
             "HOSPITAL", "TP_IDADE", "VACINA_COV", "RAIOX_RES"]}
FORMA = ["PCR_SARS2", "AN_SARS2", "EVOLUCAO", "RAIOX_RES", "AMOSTRA", "OBES_IMC"]
LITERAL = ["PCR_SARS2", "AN_SARS2"]
GATE_COLS = {c for c in S.GATES} | {p for p, _, _, _ in S.GATES.values()}
ALVO = sorted(set(DOMINIOS) | set(FORMA) | set(S.COMORBIDITIES) | GATE_COLS
              | set(S.CHECKBOXES) | set(S.NOT_CHECKBOXES) | set(S.SINTOMAS)
              | {"FATOR_RISC", "TP_IDADE", "NU_IDADE_N", "DT_SIN_PRI", "DT_NASC",
                 "SG_UF", "SEM_PRI", "DT_NOTIFIC", "SEM_NOT", "CLASSI_FIN"})

# Porcentagem com o denominador dito em voz alta. `Series.mean()` sobre um
# booleano anulável descarta os NA do denominador: (CLASSI_FIN == '5').mean()
# responde "entre os registros com etiologia preenchida", não "entre os
# registros". Em 2021 isso é 73,1% contra 70,2%, e nada na chamada denuncia
# qual das duas perguntas foi respondida.
def pct(mascara, denominador):
    return round(100 * int(mascara.sum()) / max(denominador, 1), 1)


perfil, dominio, funil, formas, idade, literal, preenchimento = [], [], [], [], [], [], {}
escada, flu, semana, gold = [], [], [], []

for ano in ANOS:
    nome = f"INFLUD{ano % 100:02d}-26-06-2025.parquet"
    caminho = DATA / nome
    baixado = not caminho.exists()
    if baixado:
        urllib.request.urlretrieve(f"{S3}/{ano}/{nome}", caminho)

    esquema = pq.ParquetFile(caminho).schema_arrow.names
    d = pq.read_table(caminho, columns=[c for c in ALVO if c in esquema]).to_pandas()
    n = len(d)
    norm = {c: S.normalise(d[c], c) for c in d.columns}

    # população e desfecho
    ev = norm["EVOLUCAO"]
    perfil.append({"ano": ano, "registros": n, "colunas": len(esquema),
                   "óbitos": int((ev == "2").sum()),
                   "letalidade %": pct(ev == "2", int(ev.isin(["1", "2"]).sum())),
                   "covid %": pct(norm["CLASSI_FIN"] == "5", n),
                   "sem desfecho %": pct(ev.isna() | (ev == "9"), n)})

    # conformidade de domínio
    for c, dom in DOMINIOS.items():
        if c in norm:
            v = norm[c]
            mau = v.notna() & ~v.isin(dom)
            dominio.append({"ano": ano, "campo": c, "fora": int(mau.sum()),
                            "exemplos": sorted(v[mau].unique())[:2]})

    # o que uma regra literal custaria
    for c in LITERAL:
        if c in d:
            ing = int((d[c].astype("string").str.strip() == "1").sum())
            cor = int((norm[c] == "1").sum())
            literal.append({"ano": ano, "campo": c, "regra literal": ing,
                            "após normalise": cor, "perdidos": cor - ing})

    # o portão do FATOR_RISC, e o preço de ignorá-lo
    g = norm["FATOR_RISC"].isin(["1", "S"])
    com = [c for c in S.COMORBIDITIES if c in norm]
    algum = pd.concat([norm[c].notna() for c in com], axis=1).any(axis=1)
    razoes = []
    for c in com:
        est = S.missing_state(norm[c], g)
        real = int((est == "ausente").sum()) / max(int(g.sum()), 1)
        if real:
            razoes.append(int(est.isin(["nao_aplicavel", "ausente"]).sum()) / n / real)
    funil.append({"ano": ano, "declara fator de risco %": pct(g, n),
                  "sem fator → alguma marcada %": round(100 * int(algum[~g].sum()) / max(int((~g).sum()), 1), 2),
                  "com fator → alguma marcada %": pct(algum[g], int(g.sum())),
                  "vazio ingênuo / ausência real": round(sum(razoes) / len(razoes), 1)})

    # idade: unidade e implausibilidade
    tp = norm["TP_IDADE"]
    anos_ = ((S.parse_dates(norm["DT_SIN_PRI"], "DT_SIN_PRI")
              - S.parse_dates(norm["DT_NASC"], "DT_NASC")).dt.days / 365.25)
    idade.append({"ano": ano, "em dias": int((tp == "1").sum()),
                  "em meses": int((tp == "2").sum()),
                  "não em anos %": round(100 * int(tp.isin(["1", "2"]).sum()) / max(n, 1), 2),
                  "idade < 0": int((anos_ < 0).sum()),
                  "idade > 120": int((anos_ > 120).sum())})

    # os números da mesa do Gold: coorte, alvos candidatos, taxas
    def is1(c):
        return (norm[c] == "1").fillna(False)

    coorte = is1("HOSPITAL") & (is1("TOSSE") | is1("GARGANTA")) & (
        is1("DISPNEIA") | is1("SATURACAO") | is1("DESC_RESP"))
    covid = is1("PCR_SARS2") | is1("AN_SARS2") | (norm["CLASSI_FIN"] == "5").fillna(False)
    fechada = coorte & ev.isin(["1", "2"])
    gold.append({"ano": ano,
                 "coorte hospitalizada": int(coorte.sum()),
                 "covid_caso": int(covid.sum()),
                 "óbito na coorte fechada %": round(100 * int((coorte & (ev == "2")).sum()) / max(int(fechada.sum()), 1), 1),
                 "UTI na coorte %": round(100 * int((coorte & is1("UTI")).sum()) / max(int(coorte.sum()), 1), 1),
                 "ventilação invasiva %": round(100 * int((coorte & (norm["SUPORT_VEN"] == "1")).sum()) / max(int(coorte.sum()), 1), 1)})

    # escada de portões: contradição re-medida por ano
    for child, (parent, values, tier, worst) in S.GATES.items():
        filho, pai = norm[child], norm[parent]
        filled = int(filho.notna().sum())
        contradiz = int((filho.notna() & ~pai.isin(list(values))).sum())
        escada.append({"ano": ano, "portao": f"{child} <= {parent}",
                       "tier": tier, "preenchidos": filled,
                       "contradicao %": round(100 * contradiz / filled, 4) if filled else 0.0})

    # recontagem influenza: definicao antiga (so valores) vs oficial
    marcados = {c: (norm[c] == "1").fillna(False) for c in S.CHECKBOXES}
    fluasu, flubli = norm["PCR_FLUASU"], norm["PCR_FLUBLI"]
    tri_a = ((norm["TP_FLU_AN"] == "1") | (norm["TP_FLU_PCR"] == "1")).fillna(False)
    tri_b = ((norm["TP_FLU_AN"] == "2") | (norm["TP_FLU_PCR"] == "2")).fillna(False)
    sub_a = fluasu.isin(["1", "2", "4", "5", "6"]).fillna(False)
    lin_b = flubli.isin(["1", "2"]).fillna(False)
    a_n_sub = (fluasu == "3").fillna(False) | (tri_a & ~sub_a)
    b_inc = tri_b & ~lin_b
    velho = (sub_a | lin_b)
    novo = velho | a_n_sub | b_inc
    flu.append({"ano": ano, "a_n_sub": int(a_n_sub.sum()), "b_inconclusivo": int(b_inc.sum()),
                "influenza_geral ANTES": int(velho.sum()), "influenza_geral OFICIAL": int(novo.sum())})

    # semana epidemiologica: ISO vs MMWR contra o gabarito publicado
    for dcol, wcol in (("DT_SIN_PRI", "SEM_PRI"), ("DT_NOTIFIC", "SEM_NOT")):
        dt = S.parse_dates(norm[dcol], dcol)
        sem = pd.to_numeric(norm[wcol], errors="coerce")
        semana.append({"ano": ano, "campo": wcol,
                       "== ISO %": round(100 * int((sem == dt.dt.isocalendar().week).sum()) / n, 2),
                       "== MMWR %": round(100 * int((sem == S.epiweek(dt)).sum()) / n, 2)})

    for c in FORMA:
        if c in d:
            formas.append({"ano": ano, "campo": c,
                           "valores crus": sorted(d[c].dropna().astype(str).str.strip().unique())[:6]})

    preenchimento[ano] = {c: round(100 * d[c].notna().mean(), 1) for c in d.columns}

    if baixado:
        caminho.unlink()
    print(f"{ano}: {n:>9,} registros")

perfil = pd.DataFrame(perfil)
print(f"\ntotal: {perfil['registros'].sum():,}")

2019:    48,961 registros


2020: 1,206,920 registros


2021: 1,745,672 registros


2022:   560,577 registros


2023:   279,453 registros


2024:   267,984 registros

total: 4,109,567


## 1. A população muda — não é o mesmo problema clínico

In [4]:
print(perfil.to_string(index=False))

 ano  registros  colunas  óbitos  letalidade %  covid %  sem desfecho %
2019      48961      194    5429          12.1      0.0             8.7
2020    1206920      194  315445          29.0     59.8             8.4
2021    1745672      194  443459          28.9     70.2            10.6
2022     560577      194   92634          19.0     42.9            10.4
2023     279453      194   24944           9.9     18.0             7.2
2024     267984      194   20728           8.6     11.6             7.3


Letalidade de 29,0% a 8,6%; COVID de 70,2% a 11,6%. Um modelo treinado sobre a
série inteira aprende, entre outras coisas, **em que ano o paciente adoeceu**.
Isso não é defeito da base — é contexto que o Gold precisa declarar.

O `pct()` acima existe por causa de um erro cometido aqui: a primeira versão
media a fração de COVID com `.mean()` sobre um booleano anulável, que **descarta
os NA do denominador**. A resposta era 73,1% para 2021 — a fração *entre os
registros com etiologia preenchida* — quando a pergunta era 70,2%. Nada na
chamada denunciava a troca; só bateu de frente com o número do README.

## 2. Deriva de forma

A serialização não é estável entre anos. Este é o defeito que motivou normalizar
antes de inferir qualquer domínio.

In [5]:
f = pd.DataFrame(formas).pivot(index="campo", columns="ano", values="valores crus")
for campo in f.index:
    print(f"\n{campo}")
    for ano in ANOS:
        v = f.loc[campo, ano]
        if isinstance(v, list):
            print(f"   {ano}: {v}")


AMOSTRA
   2019: ['1', '2', '9']
   2020: ['1', '2', '2020-11-09', '9']
   2021: ['1', '2', '2021-02-17', '2021-07-05', '2021-09-06', '9']
   2022: ['1', '2', '2022-08-01', '9']
   2023: ['1', '2', '2023-11-20', '9']
   2024: ['1', '2', '9']

AN_SARS2
   2019: ['1']
   2020: ['1.0']
   2021: ['1']
   2022: ['1']
   2023: ['1']
   2024: ['1']

EVOLUCAO
   2019: ['1', '2', '3', '9']
   2020: ['1.0', '2.0', '2020-11-24', '3.0', '9.0']
   2021: ['1', '2', '2021-03-30', '2021-07-23', '2021-09-08', '3']
   2022: ['1', '2', '2022-08-22', '3', '9']
   2023: ['1', '2', '3', '9']
   2024: ['1', '2', '3', '9']

OBES_IMC
   2019: ['0.00', '0.1', '0.10', '0.2', '0.20', '0.60']
   2020: ['0.0', '0.00', '0.03', '0.1', '0.10', '0.2']
   2021: ['0.0', '0.00', '0.03', '0.1', '0.10', '0.2']
   2022: ['0.00', '0.10', '0.20', '0.30', '0.50', '0.9']
   2023: ['0.00', '0.10', '0.20', '0.30', '0.9', '0.90']
   2024: ['0.00', '0.10', '0.20', '0.30', '1.00', '10.00']

PCR_SARS2
   2019: ['1']
   2020: ['1.0']


`RAIOX_RES` chega como `1.0000000000` nos seis anos — daí o regex ser
`^(-?\d+)\.0+$` e não `\.0$`, que nunca o tocaria. `AMOSTRA` e `EVOLUCAO`
carregam **datas** em campos codificados. E `PCR_SARS2` é `'1'` em cinco anos e
`'1.0'` em 2020:

In [6]:
print(pd.DataFrame(literal).pivot(index="campo", columns="ano", values="perdidos")
        .to_string())

ano        2019    2020  2021  2022  2023  2024
campo                                          
AN_SARS2      0   31412     0     0     0     0
PCR_SARS2     0  519518     0     0     0     0


Meio milhão de registros COVID-positivos desapareceriam de 2020 sob uma regra
literal — **sem mudar a contagem de linhas**, que é o modo mais silencioso de
destruir dado.

### Quantas das 194 colunas isso atinge

Medido sobre o [`PROFILE.json`](../PROFILE.json), gerado por
[`tools/srag_profile.py`](../../../tools/srag_profile.py), que
deliberadamente **não aceita parâmetro de ano**.

In [7]:
pj = json.loads((REPO / "modules/00-dataset/PROFILE.json").read_text())
colunas = sorted({c for a in pj.values() for c in a})
forma_muda, ausente_em_ano, fill_muda = [], [], []
for c in colunas:
    com_dado = [frozenset(pj[a][c]["shapes"]) for a in pj if c in pj[a] and pj[a][c]["shapes"]]
    vazios = [a for a in pj if c in pj[a] and not pj[a][c]["shapes"]]
    if len(set(com_dado)) > 1:
        forma_muda.append(c)
    if vazios and com_dado:
        ausente_em_ano.append(c)
    fills = [pj[a][c]["fill"] for a in pj if c in pj[a]]
    if fills and max(fills) - min(fills) > 5:
        fill_muda.append(c)
print(f"colunas perfiladas                          : {len(colunas)}")
print(f"mudam de FORMA entre anos que têm dado      : {len(forma_muda)}")
print(f"existem num ano e são 100% vazias em outro  : {len(ausente_em_ano)}")
print(f"mudam >5 p.p. de PREENCHIMENTO              : {len(fill_muda)}")

colunas perfiladas                          : 194
mudam de FORMA entre anos que têm dado      : 75
existem num ano e são 100% vazias em outro  : 21
mudam >5 p.p. de PREENCHIMENTO              : 103


São dois achados distintos, e vale não fundi-los: **75 colunas mudam de forma**
entre anos em que têm dado (é o número em [`PROFILE.md`](../PROFILE.md)), e
outras **21 nem sempre existem** — 100% vazias em algum ano. A primeira quebra
regras escritas como literal; a segunda quebra qualquer modelo que assuma um
esquema estável.

## 3. Violações de domínio, ano a ano

In [8]:
dv = pd.DataFrame(dominio)
tab = dv.pivot_table(index="campo", columns="ano", values="fora", aggfunc="sum", fill_value=0)
print(tab.loc[tab.sum(axis=1) > 0].to_string())
print("\nexemplos observados:")
print(dv[dv["fora"] > 0][["ano", "campo", "exemplos"]].to_string(index=False))

ano         2019  2020  2021  2022  2023  2024
campo                                         
CRITERIO       0     0     0     0     1     0
EVOLUCAO       0     1     3     1     0     0
HOSPITAL       0     1     2     1     0     0
SUPORT_VEN     0     0     0     1     0     0
UTI            0     0     2     1     1     0
VACINA_COV     0     0     1     0     0     0

exemplos observados:
 ano      campo                                 exemplos
2020   EVOLUCAO                             [2020-11-24]
2020   HOSPITAL                             [2020-11-20]
2021   EVOLUCAO                 [2021-03-30, 2021-07-23]
2021        UTI                 [2021-03-19, 2021-09-04]
2021   HOSPITAL                 [2021-03-15, 2021-09-04]
2021 VACINA_COV                             [23/03/2021]
2022   EVOLUCAO                             [2022-08-22]
2022        UTI                             [2022-08-01]
2022 SUPORT_VEN                                      [6]
2022   HOSPITAL                 

Datas e texto onde deveria haver código, em todos os anos. São as mesmas linhas
deslocadas em um campo, que o Silver põe em quarentena — poucas, mas presentes na
série toda, e nenhuma checagem anterior a este módulo as via.

## 4. O portão do `FATOR_RISC`, nos seis anos

Este é o número que sustenta a distinção entre `nao_aplicavel` e `ausente`. Se
falhasse num único ano, a regra estaria errada.

In [9]:
print(pd.DataFrame(funil).to_string(index=False))

 ano  declara fator de risco %  sem fator → alguma marcada %  com fator → alguma marcada %  vazio ingênuo / ausência real
2019                      42.4                           0.0                         100.0                            5.7
2020                      63.1                           0.0                         100.0                            1.8
2021                      55.8                           0.0                          99.9                            1.8
2022                      55.1                           0.0                          99.9                            1.9
2023                      45.0                           0.0                          98.5                            2.2
2024                      44.3                           0.0                          96.6                            2.3


Zero por cento em todos os seis anos, sem exceção: o sistema desabilita as treze
comorbidades quando não há fator de risco declarado. A última coluna é o preço de
ignorar isso — o fator pelo qual tratar todo vazio como ausência infla a
estatística de dado faltante.

### 4b. A escada de portões, nos seis anos

O precedente virou regra-G: contradição ≤ 0,05% em **cada** ano, com
suporte. A tabela mostra o pior ano de cada portão adotado; os reprovados
ficam em `GATES_REJECTED` com o número que os reprovou — destaque para o
portão dos checkboxes antigênicos: 0,00% em cinco anos, 6,40% em 2024.
Uma medição de um ano o teria adotado.

In [10]:
e = pd.DataFrame(escada)
pior = e.loc[e.groupby("portao")["contradicao %"].idxmax()]
pior = pior.sort_values(["tier", "contradicao %"])[["portao", "tier", "ano", "contradicao %"]]
print(pior.to_string(index=False))
print(f"\nportões adotados: {len(S.GATES)}   rejeitados com número: {len(S.GATES_REJECTED)}")

                  portao tier  ano  contradicao %
    CS_ETINIA <= CS_RACA    A 2019         0.0000
  MORB_DESC <= OUT_MORBI    A 2019         0.0000
 OBESIDADE <= FATOR_RISC    A 2019         0.0000
 OUT_ANTIV <= TP_ANTIVIR    A 2019         0.0000
 OUT_MORBI <= FATOR_RISC    A 2019         0.0000
PCR_FLUASU <= TP_FLU_PCR    A 2019         0.0000
 TP_FLU_AN <= POS_AN_FLU    A 2019         0.0000
TP_FLU_PCR <= POS_PCRFLU    A 2019         0.0000
      ASMA <= FATOR_RISC    A 2020         0.0002
CARDIOPATI <= FATOR_RISC    A 2020         0.0002
  DIABETES <= FATOR_RISC    A 2020         0.0002
HEMATOLOGI <= FATOR_RISC    A 2020         0.0002
  HEPATICA <= FATOR_RISC    A 2020         0.0002
IMUNODEPRE <= FATOR_RISC    A 2020         0.0002
NEUROLOGIC <= FATOR_RISC    A 2020         0.0002
PNEUMOPATI <= FATOR_RISC    A 2020         0.0002
     RENAL <= FATOR_RISC    A 2020         0.0002
 SIND_DOWN <= FATOR_RISC    A 2020         0.0002
  PUERPERA <= FATOR_RISC    A 2023         0.0012


As contradições de milésimos nos portões tier A são as **linhas em
quarentena**: este caderno mede o Bronze, antes da remoção — a comorbidade
"preenchida" com `FATOR_RISC` inválido em 2020/2023 é a linha deslocada cujo
`FATOR_RISC` carrega `CORIZA"`. No Silver, pós-quarentena, o `build()`
re-mede e exige o limite.

## 5. Deriva de preenchimento — a disponibilidade codifica o ano

In [11]:
pf = pd.DataFrame(preenchimento).reindex(sorted(preenchimento[2021]))
pf["amplitude"] = pf.max(axis=1) - pf.min(axis=1)
print(pf.sort_values("amplitude", ascending=False).head(12).round(1).to_string())

            2019  2020  2021  2022   2023   2024  amplitude
VACINA_COV   1.3  13.2  75.1  99.8  100.0  100.0       98.7
FADIGA       0.6  46.4  69.5  67.5   72.6   76.4       75.8
TRAT_COV     0.2   1.0   1.1  17.8   75.1   75.6       75.4
DOR_ABD      0.3  45.1  65.2  65.0   70.6   74.8       74.5
PERD_OLFT    0.3  45.6  66.0  64.1   69.7   74.0       73.7
PERD_PALA    0.3  45.5  66.0  64.0   69.7   73.9       73.6
DT_ANTIVIR  60.6  14.0   1.9   3.1    5.4    7.6       58.7
TP_ANTIVIR  60.6  14.3   2.3   3.4    5.9    8.2       58.3
DT_RAIOX    74.1  29.4  18.8  24.8   36.5   41.2       55.3
PCR_SARS2    0.0  43.0  40.3  19.0    8.0    6.0       43.0
RAIOX_RES   92.2  66.2  57.5  59.2   66.0   69.3       34.7
DIARREIA    97.0  74.3  67.5  65.9   71.5   75.7       31.1


`VACINA_COV` é o caso extremo: 1,3% preenchido em 2019, 100% em 2023. Um campo
que só passa a existir a partir de certo ano é um relógio disfarçado de variável
clínica.

Para os módulos de interpretabilidade este é o caso mais interessante da base: o
modelo **vai** encontrar essa estrutura, e a explicação vai mostrá-la. É a
diferença entre uma explicação que revela algo clínico e uma que revela o
calendário do formulário.

## 6. Idade: a unidade importa, e varia com o ano

In [12]:
print(pd.DataFrame(idade).to_string(index=False))

 ano  em dias  em meses  não em anos %  idade < 0  idade > 120
2019     1792     12875          29.96          0            0
2020     4226     19648           1.98          0            7
2021     6429     45930           3.00          0            5
2022     4344     56224          10.80          0            3
2023     3379     62834          23.69          1            3
2024     3808     63133          24.98          0            0


A fração que **não** está em anos vai de 2,0% a 30,0% conforme o ano — ler
`NU_IDADE_N` sem `TP_IDADE` erra de um jeito que varia com o ano.

Idades impossíveis são raras e **não** aparecem em todos os anos: negativas só em
2023, acima de 120 em quatro dos seis. Uma auditoria anterior tinha reportado
zero para ambas, por medir só em 2021. O Silver as registra sem corrigir — não se
sabe qual das duas datas está errada.

## 7. Os números que a mesa do Gold precisa

O [`GOLD.md`](../GOLD.md) é um cardápio de decisões — alvo, coorte, recorte,
split — e cada opção precisa do seu número. São estes, impressos aqui para
que o cardápio nunca cite um valor sem célula:

In [13]:
g = pd.DataFrame(gold)
print(g.to_string(index=False))
print(f"\ncoorte hospitalizada total: {g['coorte hospitalizada'].sum():,} "
      f"({100 * g['coorte hospitalizada'].sum() / perfil['registros'].sum():.1f}% dos registros)")

 ano  coorte hospitalizada  covid_caso  óbito na coorte fechada %  UTI na coorte %  ventilação invasiva %
2019                 43986          20                       11.1             34.8                   18.8
2020                696048      723677                       27.2             29.6                   14.9
2021               1032963     1229260                       27.2             29.9                   15.8
2022                298812      241976                       16.6             26.9                   10.9
2023                177987       51003                        7.0             26.2                    9.3
2024                178900       31986                        6.7             27.6                    9.6

coorte hospitalizada total: 2,428,696 (59.1% dos registros)


Três candidatos a alvo e as suas taxas na coorte hospitalizada: óbito
(27,2% → 6,7% — o regime muda o problema), admissão em UTI (26–35%),
ventilação invasiva (9–19%). O `covid_caso` mostra o recorte: 723 mil em
2020, 32 mil em 2024.

## 8. A recontagem de influenza, nos seis anos

`influenza_a_n_sub` (PCR_FLUASU == 3, ou triagem TP_FLU_* positiva sem
subtipo) e `influenza_b_inconclusivo` faltavam na primeira transcrição do
catálogo — exatamente os dois campos que `NOT_CHECKBOXES` nomeava sem
consumir.

In [14]:
f = pd.DataFrame(flu)
f["recuperados"] = f["influenza_geral OFICIAL"] - f["influenza_geral ANTES"]
print(f.to_string(index=False))
t = f[["influenza_geral ANTES", "influenza_geral OFICIAL"]].sum()
print(f"\ntotal: {t.iloc[0]:,} antes -> {t.iloc[1]:,} na definição oficial "
      f"({t.iloc[1] / t.iloc[0]:.1f}x)")

 ano  a_n_sub  b_inconclusivo  influenza_geral ANTES  influenza_geral OFICIAL  recuperados
2019      753             764                   5370                     6885         1515
2020      536             732                   1118                     2384         1266
2021     5666             262                   6174                    12096         5922
2022     4822             433                   6963                    12214         5251
2023     4398            4646                   3959                    12978         9019
2024    11955            3240                  10084                    25251        15167

total: 33,668 antes -> 71,808 na definição oficial (2.1x)


A tabela do script oficial dá 6.885 / 2.384 / 12.096 / 12.214 / 12.978 /
25.251 — os mesmos inteiros, verificados de forma independente por um segundo
cálculo a partir das colunas cruas, antes de o código ser corrigido.

## 9. A semana epidemiológica, nos seis anos

O gabarito é publicado pela própria base: `SEM_PRI` e `SEM_NOT`.

In [15]:
w = pd.DataFrame(semana).pivot(index="ano", columns="campo",
                               values=["== ISO %", "== MMWR %"])
print(w.to_string())

      == ISO %         == MMWR %        
campo  SEM_NOT SEM_PRI   SEM_NOT SEM_PRI
ano                                     
2019     95.31   85.98     100.0   100.0
2020     91.39   86.48     100.0   100.0
2021     91.22   86.33     100.0   100.0
2022     92.18   85.82     100.0   100.0
2023     93.00   85.62     100.0   100.0
2024     93.33   85.60     100.0   100.0


ISO acerta ~86% (`SEM_PRI`) e ~92% (`SEM_NOT`); MMWR acerta **100,00% nas
doze células**. A virada de ano também é MMWR: dezembro em semana 1 pertence
ao ano seguinte, janeiro em semana 52/53 ao anterior — verificado em 654 +
18.735 linhas de fronteira, sem exceção.

## 10. As colunas year-gated

100% vazias em pelo menos um ano: o vazio codifica o ano, não o paciente.
A lista vive em `S.YEAR_GATED`; o gerador do contrato a re-deriva do
PROFILE.json e **quebra o build** se divergir.

In [16]:
import json
pj = json.loads((REPO / "modules/00-dataset/PROFILE.json").read_text())
linhas = []
for c in sorted(S.YEAR_GATED):
    vazios = sorted(a for a in pj if pj[a][c]["fill"] == 0.0)
    com = sorted(set(pj) - set(vazios))
    linhas.append({"coluna": c, "anos vazios": ", ".join(vazios),
                   "primeiro ano com dado": com[0] if com else "—"})
print(pd.DataFrame(linhas).to_string(index=False))

    coluna                  anos vazios primeiro ano com dado
 CO_PS_VGM             2022, 2023, 2024                  2019
 DT_RT_VGM                   2023, 2024                  2019
DT_TRT_COV                         2019                  2020
    DT_VGM             2022, 2023, 2024                  2019
 LO_PS_VGM             2022, 2023, 2024                  2019
  OUT_ANIM                         2019                  2020
   OUT_SOR                         2019                  2020
  OUT_TRAT                         2019                  2020
  PAIS_VGM                   2022, 2024                  2019
   SOR_OUT                         2019                  2020
     TABAG             2019, 2023, 2024                  2020
 TIPO_TRAT                         2019                  2020
 VG_CODEST                         2019                  2020
  VG_DTRES                         2019                  2020
    VG_ENC                         2019                  2020
    VG_L

## 11. O que este caderno não cobre

- ~140 colunas ainda sem regra de família própria — laboratório, vacinação,
  vigilância genômica. Têm perfil e preenchimento medidos, nada mais.
- Os ~84 predicados de habilitação do dicionário; só o do `FATOR_RISC` está
  implementado.
- Integridade referencial de códigos IBGE e CNES contra tabela externa.
- Tudo que é decisão de tarefa — alvo, vazamento, recorte, split — que é do
  Gold, listado no [walkthrough §7](srag_silver_walkthrough.ipynb). A lista
  de exclusão por vazamento agora é **gerável**: classe `leakage` no
  [`COLUMNS.md`](../COLUMNS.md).